In [1]:
# Cell 1: clone repo
!git clone https://github.com/nnzhan/Graph-WaveNet.git
%cd Graph-WaveNet

Cloning into 'Graph-WaveNet'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 53 (delta 12), reused 12 (delta 12), pack-reused 33 (from 1)
Receiving objects: 100% (53/53), 267.08 KiB | 3.71 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Graph-WaveNet


In [2]:
# Clone DCRNN để lấy file adj
!git clone https://github.com/liyaguang/DCRNN.git

# Kiểm tra file có ở đó không
!ls DCRNN/data/sensor_graph/

Cloning into 'DCRNN'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 334 (delta 51), reused 34 (delta 34), pack-reused 268 (from 2)
Receiving objects: 100% (334/334), 127.90 MiB | 39.71 MiB/s, done.
Resolving deltas: 100% (153/153), done.
adj_mx_bay.pkl		graph_sensor_ids.txt
adj_mx.pkl		graph_sensor_locations_bay.csv
distances_bay_2017.csv	graph_sensor_locations.csv
distances_la_2012.csv


In [3]:
# Copy file adj vào đúng chỗ Graph-WaveNet cần
!mkdir -p /kaggle/working/Graph-WaveNet/data/sensor_graph

# METR-LA
!cp DCRNN/data/sensor_graph/adj_mx.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx.pkl

# PEMS-BAY
!cp DCRNN/data/sensor_graph/adj_mx_bay.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx_bay.pkl

In [4]:
# Kiểm tra lại
!ls /kaggle/working/Graph-WaveNet/data/sensor_graph/

adj_mx_bay.pkl	adj_mx.pkl


In [5]:
%%writefile /kaggle/working/Graph-WaveNet/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# [TUNING] Các hyperparameter từ adaptive_adjency.py
EMB_DIM          = 4   # [OPT-1] chiều embedding low-rank (gốc = 10)
TOPK             = 10  # [OPT-2] số cạnh giữ lại mỗi node
ADJ_UPDATE_FREQ  = 5   # [OPT-3] tính lại Ã mỗi N bước
# ---------------------------------------------------------------------------


# =============================================================================
# Helpers từ transformer.py
# =============================================================================

def _causal_window_mask(seq_len: int, window: int, device) -> torch.Tensor:
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
    for i in range(seq_len):
        lo = max(0, i - window + 1)
        mask[i, lo : i + 1] = False
    return mask


class RelativePositionalEncoding(nn.Module):
    def __init__(self, num_heads: int, max_len: int = 64):
        super().__init__()
        self.num_heads = num_heads
        self.rel_bias = nn.Embedding(max_len, num_heads)
        nn.init.zeros_(self.rel_bias.weight)

    def forward(self, seq_len: int) -> torch.Tensor:
        device = self.rel_bias.weight.device
        idx  = torch.arange(seq_len, device=device)
        dist = (idx.unsqueeze(1) - idx.unsqueeze(0)).clamp(min=0)
        dist = dist.clamp(max=self.rel_bias.num_embeddings - 1)
        bias = self.rel_bias(dist)          # (T, T, H)
        return bias.permute(2, 0, 1)        # (H, T, T)


class CausalWindowAttnTCN(nn.Module):
    """Causal Window Multi-Head Attention — thay thế dilated conv."""
    def __init__(self, in_channels, out_channels,
                 kernel_size=2, num_heads=4, dropout=0.1):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.window_size  = max(2, 2 * kernel_size)

        head_dim       = max(out_channels // num_heads, 1)
        self.num_heads = num_heads
        self.head_dim  = head_dim
        self.scale     = math.sqrt(head_dim)

        self.q_proj  = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.k_proj  = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.v_proj  = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * head_dim, out_channels)

        self.rel_pe  = RelativePositionalEncoding(num_heads)
        self.dropout = nn.Dropout(dropout)
        self.norm    = nn.LayerNorm(out_channels)
        self.gate    = nn.Linear(out_channels, out_channels)

        self.residual_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, N, T) → (B, out_channels, N, T-1)"""
        B, C, N, T = x.shape
        x_bn = x.permute(0, 2, 3, 1).reshape(B * N, T, C)

        Q = self.q_proj(x_bn)
        K = self.k_proj(x_bn)
        V = self.v_proj(x_bn)

        H, D = self.num_heads, self.head_dim
        Q = Q.view(B * N, T, H, D).transpose(1, 2)
        K = K.view(B * N, T, H, D).transpose(1, 2)
        V = V.view(B * N, T, H, D).transpose(1, 2)

        attn     = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        rel_bias = self.rel_pe(T)
        attn     = attn + rel_bias.unsqueeze(0)

        mask = _causal_window_mask(T, self.window_size, x.device)
        attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).reshape(B * N, T, H * D)
        out = self.out_proj(out)
        out = out * torch.sigmoid(self.gate(out))
        out = self.norm(out)

        out = out.view(B, N, T, -1).permute(0, 3, 1, 2)
        res = self.residual_proj(x)
        out = out[:, :, :, 1:] + res[:, :, :, 1:]
        return out


class SkipAggregationAttn(nn.Module):
    """Cross-layer Skip Attention — aggregate skip connections."""
    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        while channels % num_heads != 0 and num_heads > 1:
            num_heads //= 2
        self.attn = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads,
            batch_first=True, dropout=0.1,
        )
        self.norm = nn.LayerNorm(channels)

    def forward(self, skip_list: list) -> torch.Tensor:
        processed = []
        for s in skip_list:
            if s.dim() == 4:
                s = s.mean(dim=-1, keepdim=True)
            processed.append(s)

        B, C, N, _ = processed[0].shape
        elems   = [p.squeeze(-1).permute(0, 2, 1) for p in processed]  # each (B,N,C)
        stacked = torch.stack(elems, dim=2)          # (B, N, K, C)
        K       = stacked.shape[2]
        stacked_2d = stacked.view(B * N, K, C)

        attended, _ = self.attn(stacked_2d, stacked_2d, stacked_2d)
        attended    = self.norm(attended + stacked_2d)

        out = attended.mean(dim=1)                   # (BN, C)
        out = out.view(B, N, C).permute(0, 2, 1)
        return out.unsqueeze(-1)                     # (B, C, N, 1)


# =============================================================================
# GCN (giữ nguyên bản gốc)
# =============================================================================

class nconv(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, A):
        x = torch.einsum('ncvl,vw->ncwl', (x, A))
        return x.contiguous()


class linear(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Conv2d(c_in, c_out, kernel_size=(1, 1),
                             padding=(0, 0), stride=(1, 1), bias=True)

    def forward(self, x):
        return self.mlp(x)


class gcn(nn.Module):
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self.nconv = nconv()
        c_in_actual = (order * support_len + 1) * c_in
        self.mlp    = linear(c_in_actual, c_out)
        self.dropout = dropout
        self.order   = order

    def forward(self, x, support):
        out = [x]
        for a in support:
            x1 = self.nconv(x, a)
            out.append(x1)
            for _ in range(2, self.order + 1):
                x2 = self.nconv(x1, a)
                out.append(x2)
                x1 = x2
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h


# =============================================================================
# Main model
# =============================================================================

class gwnet(nn.Module):
    """
    Ablation A: WITHOUT Dynamic Adjacency.

    Thay DynamicAdaptiveAdj bằng static adaptive adj với:
      [OPT-1] Low-rank embedding  (emb_dim < 10)
      [OPT-2] Top-k sparsification
      [OPT-3] Adjacency caching

    Interface giữ nguyên bản gốc.
    """
    def __init__(
        self,
        device,
        num_nodes: int,
        dropout: float = 0.3,
        supports=None,
        gcn_bool: bool = True,
        addaptadj: bool = True,
        aptinit=None,
        in_dim: int = 2,
        out_dim: int = 12,
        residual_channels: int = 32,
        dilation_channels: int = 32,
        skip_channels: int = 256,
        end_channels: int = 512,
        kernel_size: int = 2,
        blocks: int = 4,
        layers: int = 2,
        # [OPT-1,2,3] từ adaptive_adjency.py
        emb_dim: int = EMB_DIM,
        topk: int = TOPK,
        adj_update_freq: int = ADJ_UPDATE_FREQ,
    ):
        super().__init__()

        self.dropout    = dropout
        self.blocks     = blocks
        self.layers     = layers
        self.gcn_bool   = gcn_bool
        self.addaptadj  = addaptadj
        self.supports   = supports

        # [OPT-2] top-k
        self.topk = topk

        # [OPT-3] adjacency cache
        self._adj_step_counter    = 0
        self.adj_update_freq      = adj_update_freq
        self._cached_adp          = None
        self._cached_new_supports = None

        # ── [OPT-1] Low-rank adaptive adjacency ──────────────────────────
        self.supports_len = 0 if supports is None else len(supports)

        if gcn_bool and addaptadj:
            if supports is None:
                self.supports = []
            if aptinit is None:
                # [OPT-1] dùng emb_dim thay vì hardcode 10
                self.nodevec1 = nn.Parameter(
                    torch.randn(num_nodes, emb_dim).to(device), requires_grad=True
                )
                self.nodevec2 = nn.Parameter(
                    torch.randn(emb_dim, num_nodes).to(device), requires_grad=True
                )
            else:
                m, p, n = torch.svd(aptinit)
                initemb1 = torch.mm(m[:, :emb_dim], torch.diag(p[:emb_dim] ** 0.5))
                initemb2 = torch.mm(torch.diag(p[:emb_dim] ** 0.5), n[:, :emb_dim].t())
                self.nodevec1 = nn.Parameter(initemb1.to(device), requires_grad=True)
                self.nodevec2 = nn.Parameter(initemb2.to(device), requires_grad=True)
            self.supports_len += 1

        # ── Layers ───────────────────────────────────────────────────────
        self.start_conv = nn.Conv2d(
            in_channels=in_dim, out_channels=residual_channels,
            kernel_size=(1, 1)
        )
        self.tcn_layers      = nn.ModuleList()
        self.gcn_layers      = nn.ModuleList()
        self.skip_convs      = nn.ModuleList()
        self.residual_convs  = nn.ModuleList()
        self.bn              = nn.ModuleList()

        receptive_field = 1
        for b in range(blocks):
            new_dilation = 1
            for _ in range(layers):
                self.tcn_layers.append(
                    CausalWindowAttnTCN(
                        in_channels=residual_channels,
                        out_channels=dilation_channels,
                        kernel_size=new_dilation,
                        num_heads=4,
                        dropout=dropout,
                    )
                )
                self.skip_convs.append(
                    nn.Conv2d(dilation_channels, skip_channels, kernel_size=(1, 1))
                )
                if gcn_bool:
                    self.gcn_layers.append(
                        gcn(dilation_channels, residual_channels,
                            dropout, support_len=self.supports_len)
                    )
                else:
                    self.residual_convs.append(
                        nn.Conv2d(dilation_channels, residual_channels,
                                  kernel_size=(1, 1))
                    )
                self.bn.append(nn.BatchNorm2d(residual_channels))
                receptive_field += new_dilation
                new_dilation    *= 2

        self.receptive_field = receptive_field

        # FIX-01: Cross-layer Skip Attention
        self.skip_agg_attn = SkipAggregationAttn(
            channels=skip_channels, num_heads=8
        )

        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels,
                                    kernel_size=(1, 1), bias=True)
        self.end_conv_2 = nn.Conv2d(end_channels, out_dim,
                                    kernel_size=(1, 1), bias=True)

    # ------------------------------------------------------------------
    # [OPT-2] Top-k sparsification
    # ------------------------------------------------------------------
    def _topk_sparse(self, adp):
        k = min(self.topk, adp.size(1))
        topk_vals, topk_idx = torch.topk(adp, k, dim=1)
        mask = torch.zeros_like(adp)
        mask.scatter_(1, topk_idx, 1.0)
        return adp * mask

    # ------------------------------------------------------------------
    # [OPT-3] Tính và cache adjacency (detached)
    # ------------------------------------------------------------------
    def _compute_and_cache_adj(self):
        adp_full   = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
        adp_sparse = self._topk_sparse(adp_full)
        adp_cached = adp_sparse.detach()
        self._cached_adp          = adp_cached
        self._cached_new_supports = self.supports + [adp_cached]

    def forward(self, input):
        in_len = input.size(3)
        if in_len < self.receptive_field:
            input = F.pad(input, (self.receptive_field - in_len, 0, 0, 0))

        x = self.start_conv(input)

        # ── Static adaptive adj + Top-k + Cache (OPT-1,2,3) ─────────────
        new_supports = None
        if self.gcn_bool and self.addaptadj and self.supports is not None:
            if self.training:
                should_update = (
                    self._cached_new_supports is None or
                    self._adj_step_counter % self.adj_update_freq == 0
                )
                if should_update:
                    adp_full  = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
                    adp_live  = self._topk_sparse(adp_full)   # có gradient
                    self._compute_and_cache_adj()              # lưu detached
                    new_supports = self.supports + [adp_live]
                else:
                    new_supports = self._cached_new_supports
                self._adj_step_counter += 1
            else:
                adp_full   = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
                adp_sparse = self._topk_sparse(adp_full)
                new_supports = self.supports + [adp_sparse.detach()]

        # ── ST layers ────────────────────────────────────────────────────
        skip_list = []
        gcn_idx   = 0
        for layer_idx in range(self.blocks * self.layers):
            residual = x

            x_tcn = self.tcn_layers[layer_idx](x)

            s = self.skip_convs[layer_idx](x_tcn)
            skip_list.append(s)

            if self.gcn_bool and self.supports is not None:
                if self.addaptadj:
                    x = self.gcn_layers[gcn_idx](x_tcn, new_supports)
                else:
                    x = self.gcn_layers[gcn_idx](x_tcn, self.supports)
                gcn_idx += 1
            else:
                x = self.residual_convs[layer_idx](x_tcn)

            x = x + residual[:, :, :, -x.size(3):]
            x = self.bn[layer_idx](x)

        # ── Cross-layer Skip Attention (FIX-01) ──────────────────────────
        x = self.skip_agg_attn(skip_list)   # (B, skip_channels, N, 1)

        x = F.relu(x)
        x = F.relu(self.end_conv_1(x))
        x = self.end_conv_2(x)
        return x

Overwriting /kaggle/working/Graph-WaveNet/model.py


In [6]:
%%writefile /kaggle/working/Graph-WaveNet/train.py
import torch
import numpy as np
import argparse
import time
import util
import matplotlib.pyplot as plt
from engine import trainer

parser = argparse.ArgumentParser()
parser.add_argument('--device',type=str,default='cuda:3',help='')
parser.add_argument('--data',type=str,default='data/METR-LA',help='data path')
parser.add_argument('--adjdata',type=str,default='data/sensor_graph/adj_mx.pkl',help='adj data path')
parser.add_argument('--adjtype',type=str,default='doubletransition',help='adj type')
parser.add_argument('--gcn_bool',action='store_true',help='whether to add graph convolution layer')
parser.add_argument('--aptonly',action='store_true',help='whether only adaptive adj')
parser.add_argument('--addaptadj',action='store_true',help='whether add adaptive adj')
parser.add_argument('--randomadj',action='store_true',help='whether random initialize adaptive adj')
parser.add_argument('--seq_length',type=int,default=12,help='')
parser.add_argument('--nhid',type=int,default=32,help='')
parser.add_argument('--in_dim',type=int,default=2,help='inputs dimension')
parser.add_argument('--num_nodes',type=int,default=207,help='number of nodes')
parser.add_argument('--batch_size',type=int,default=64,help='batch size')
parser.add_argument('--learning_rate',type=float,default=0.001,help='learning rate')
parser.add_argument('--dropout',type=float,default=0.3,help='dropout rate')
parser.add_argument('--weight_decay',type=float,default=0.0001,help='weight decay rate')
parser.add_argument('--epochs',type=int,default=100,help='')
parser.add_argument('--print_every',type=int,default=50,help='')
#parser.add_argument('--seed',type=int,default=99,help='random seed')
parser.add_argument('--save',type=str,default='./garage/metr',help='save path')
parser.add_argument('--expid',type=int,default=1,help='experiment id')
parser.add_argument('--start_epoch', type=int, default=1)
parser.add_argument('--checkpoint', type=str, default=None)
args = parser.parse_args()




def main():
    #set seed
    #torch.manual_seed(args.seed)
    #np.random.seed(args.seed)
    #load data
    device = torch.device(args.device)
    sensor_ids, sensor_id_to_ind, adj_mx = util.load_adj(args.adjdata,args.adjtype)
    dataloader = util.load_dataset(args.data, args.batch_size, args.batch_size, args.batch_size)
    scaler = dataloader['scaler']
    supports = [torch.tensor(i).to(device) for i in adj_mx]

    print(args)

    if args.randomadj:
        adjinit = None
    else:
        adjinit = supports[0]

    if args.aptonly:
        supports = None

    engine = trainer(scaler, args.in_dim, args.seq_length, args.num_nodes, args.nhid, args.dropout,
                     args.learning_rate, args.weight_decay, device, supports, args.gcn_bool, args.addaptadj,
                     adjinit)

    # ✅ Load SAU khi tạo engine
    if args.checkpoint:
        engine.model.load_state_dict(torch.load(args.checkpoint))
        print(f'Loaded checkpoint: {args.checkpoint}')


    print("start training...",flush=True)
    his_loss =[]
    val_time = []
    train_time = []
    for i in range(args.start_epoch, args.epochs + 1):
        #if i % 10 == 0:
            #lr = max(0.000002,args.learning_rate * (0.1 ** (i // 10)))
            #for g in engine.optimizer.param_groups:
                #g['lr'] = lr
        train_loss = []
        train_mape = []
        train_rmse = []
        t1 = time.time()
        dataloader['train_loader'].shuffle()
        for iter, (x, y) in enumerate(dataloader['train_loader'].get_iterator()):
            trainx = torch.Tensor(x).to(device)
            trainx= trainx.transpose(1, 3)
            trainy = torch.Tensor(y).to(device)
            trainy = trainy.transpose(1, 3)
            metrics = engine.train(trainx, trainy[:,0,:,:])
            train_loss.append(metrics[0])
            train_mape.append(metrics[1])
            train_rmse.append(metrics[2])
            if iter % args.print_every == 0 :
                log = 'Iter: {:03d}, Train Loss: {:.4f}, Train MAPE: {:.4f}, Train RMSE: {:.4f}'
                print(log.format(iter, train_loss[-1], train_mape[-1], train_rmse[-1]),flush=True)
        t2 = time.time()
        train_time.append(t2-t1)
        #validation
        valid_loss = []
        valid_mape = []
        valid_rmse = []


        s1 = time.time()
        for iter, (x, y) in enumerate(dataloader['val_loader'].get_iterator()):
            testx = torch.Tensor(x).to(device)
            testx = testx.transpose(1, 3)
            testy = torch.Tensor(y).to(device)
            testy = testy.transpose(1, 3)
            metrics = engine.eval(testx, testy[:,0,:,:])
            valid_loss.append(metrics[0])
            valid_mape.append(metrics[1])
            valid_rmse.append(metrics[2])
        s2 = time.time()
        log = 'Epoch: {:03d}, Inference Time: {:.4f} secs'
        print(log.format(i,(s2-s1)))
        val_time.append(s2-s1)
        mtrain_loss = np.mean(train_loss)
        mtrain_mape = np.mean(train_mape)
        mtrain_rmse = np.mean(train_rmse)

        mvalid_loss = np.mean(valid_loss)
        mvalid_mape = np.mean(valid_mape)
        mvalid_rmse = np.mean(valid_rmse)
        his_loss.append(mvalid_loss)

        log = 'Epoch: {:03d}, Train Loss: {:.4f}, Train MAPE: {:.4f}, Train RMSE: {:.4f}, Valid Loss: {:.4f}, Valid MAPE: {:.4f}, Valid RMSE: {:.4f}, Training Time: {:.4f}/epoch'
        print(log.format(i, mtrain_loss, mtrain_mape, mtrain_rmse, mvalid_loss, mvalid_mape, mvalid_rmse, (t2 - t1)),flush=True)
        torch.save(engine.model.state_dict(), args.save+"_epoch_"+str(i)+"_"+str(round(mvalid_loss,2))+".pth")
    print("Average Training Time: {:.4f} secs/epoch".format(np.mean(train_time)))
    print("Average Inference Time: {:.4f} secs".format(np.mean(val_time)))

    #testing
    bestid = np.argmin(his_loss)
    engine.model.load_state_dict(torch.load(args.save+"_epoch_"+str(bestid+1)+"_"+str(round(his_loss[bestid],2))+".pth"))


    outputs = []
    realy = torch.Tensor(dataloader['y_test']).to(device)
    realy = realy.transpose(1,3)[:,0,:,:]

    for iter, (x, y) in enumerate(dataloader['test_loader'].get_iterator()):
        testx = torch.Tensor(x).to(device)
        testx = testx.transpose(1,3)
        with torch.no_grad():
            preds = engine.model(testx).transpose(1,3)
        outputs.append(preds.squeeze())

    yhat = torch.cat(outputs,dim=0)
    yhat = yhat[:realy.size(0),...]


    print("Training finished")
    print("The valid loss on best model is", str(round(his_loss[bestid],4)))


    amae = []
    amape = []
    armse = []
    for i in range(12):
        pred = scaler.inverse_transform(yhat[:,:,i])
        real = realy[:,:,i]
        metrics = util.metric(pred,real)
        log = 'Evaluate best model on test data for horizon {:d}, Test MAE: {:.4f}, Test MAPE: {:.4f}, Test RMSE: {:.4f}'
        print(log.format(i+1, metrics[0], metrics[1], metrics[2]))
        amae.append(metrics[0])
        amape.append(metrics[1])
        armse.append(metrics[2])

    log = 'On average over 12 horizons, Test MAE: {:.4f}, Test MAPE: {:.4f}, Test RMSE: {:.4f}'
    print(log.format(np.mean(amae),np.mean(amape),np.mean(armse)))
    torch.save(engine.model.state_dict(), args.save+"_exp"+str(args.expid)+"_best_"+str(round(his_loss[bestid],2))+".pth")



if __name__ == "__main__":
    t1 = time.time()
    main()
    t2 = time.time()
    print("Total time spent: {:.4f}".format(t2-t1))

Overwriting /kaggle/working/Graph-WaveNet/train.py


In [7]:
!pip install -r requirements.txt

In [8]:
!rm -rf data/METR-LA data/PEMS-BAY

!python generate_training_data.py \
    --output_dir=data/METR-LA \
    --traffic_df_filename=/kaggle/input/datasets/annnnguyen/metr-la-dataset/METR-LA.h5

!python generate_training_data.py \
    --output_dir=data/PEMS-BAY \
    --traffic_df_filename=/kaggle/input/datasets/scchuy/pemsbay/pems-bay.h5

x shape:  (34249, 12, 207, 2) , y shape:  (34249, 12, 207, 2)
train x:  (23974, 12, 207, 2) y: (23974, 12, 207, 2)
val x:  (3425, 12, 207, 2) y: (3425, 12, 207, 2)
test x:  (6850, 12, 207, 2) y: (6850, 12, 207, 2)
x shape:  (52093, 12, 325, 2) , y shape:  (52093, 12, 325, 2)
train x:  (36465, 12, 325, 2) y: (36465, 12, 325, 2)
val x:  (5209, 12, 325, 2) y: (5209, 12, 325, 2)
test x:  (10419, 12, 325, 2) y: (10419, 12, 325, 2)


In [9]:
# Tạo thư mục lưu checkpoint trước
!mkdir -p garage

In [10]:
# !python train.py --device cuda:0 --data data/PEMS-BAY --adjdata data/sensor_graph/adj_mx_bay.pkl --gcn_bool --addaptadj --num_nodes 325 --checkpoint /kaggle/input/graohwavenet/_epoch_76_1.66.pth --start_epoch 77 --epochs 100

In [11]:
# !python train.py \
#   --device cuda:0 \
#   --data data/METR-LA \
#   --gcn_bool --addaptadj \
#   --checkpoint /kaggle/input/graohwavenet/_epoch_76_1.66.pth \
#   --start_epoch 77 \

In [12]:
# !python train.py \
#     --device cuda:0 \
#     --data data/PEMS-BAY \
#     --adjdata data/sensor_graph/adj_mx_bay.pkl \
#     --adjtype doubletransition \
#     --gcn_bool \
#     --addaptadj \
#     --num_nodes 325 \
#     --save garage/ \
#     --expid 2

In [13]:
!python test.py \
    --device cuda:0 \
    --data data/PEMS-BAY \
    --adjdata data/sensor_graph/adj_mx_bay.pkl \
    --adjtype doubletransition \
    --gcn_bool \
    --addaptadj \
    --num_nodes 325 \
    --checkpoint /kaggle/input/graohwavenet/_epoch_73_1.63.pth

model load successfully
Evaluate best model on test data for horizon 1, Test MAE: 0.9137, Test MAPE: 0.0184, Test RMSE: 1.6483
Evaluate best model on test data for horizon 2, Test MAE: 1.1600, Test MAPE: 0.0242, Test RMSE: 2.2937
Evaluate best model on test data for horizon 3, Test MAE: 1.3373, Test MAPE: 0.0288, Test RMSE: 2.8027
Evaluate best model on test data for horizon 4, Test MAE: 1.4711, Test MAPE: 0.0326, Test RMSE: 3.1967
Evaluate best model on test data for horizon 5, Test MAE: 1.5726, Test MAPE: 0.0355, Test RMSE: 3.4955
Evaluate best model on test data for horizon 6, Test MAE: 1.6545, Test MAPE: 0.0379, Test RMSE: 3.7303
Evaluate best model on test data for horizon 7, Test MAE: 1.7205, Test MAPE: 0.0398, Test RMSE: 3.9132
Evaluate best model on test data for horizon 8, Test MAE: 1.7764, Test MAPE: 0.0413, Test RMSE: 4.0603
Evaluate best model on test data for horizon 9, Test MAE: 1.8242, Test MAPE: 0.0427, Test RMSE: 4.1802
Evaluate best model on test data for horizon 10, 